# Sudoku AI — Model Evaluation
Tests all 6 trained models on the held-out test set.  
Metrics: **Cell Accuracy**, **Puzzle Accuracy**, **F1 Score** (per digit + macro/weighted), **Confusion Matrix**.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../backend'))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import json
import pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset

from models.mlp_solver    import MLPSolver
from models.rnn_solver    import RNNSolver
from models.lstm_solver   import LSTMSolver
from models.gru_solver    import GRUSolver
from models.hybrid_solver import HybridSolver

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WEIGHTS_DIR = Path('weights')
DATA_PATH   = Path('../backend/data/generated_test.npz')
BATCH_SIZE  = 512

print(f'Device : {DEVICE}')
print(f'Test   : {DATA_PATH}')

## 1 — Legacy CNN (matches saved weights)

In [ ]:
class LegacyCNNSolver(nn.Module):
    """
    Plain sequential CNN matching the architecture saved in cnn.pt.
    8x (Conv2d → BatchNorm2d → ReLU) + Dropout + final 1×1 conv.
    """
    def __init__(self, channels: int = 64, num_blocks: int = 8, dropout: float = 0.2):
        super().__init__()
        layers = [
            nn.Conv2d(10, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
        ]
        for _ in range(num_blocks - 1):
            layers += [
                nn.Conv2d(channels, channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(channels),
                nn.ReLU(),
            ]
        layers.append(nn.Dropout(p=dropout))         # index 24 in saved weights
        layers.append(nn.Conv2d(channels, 9, kernel_size=1))  # index 25
        self.cnn = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B = x.size(0)
        x_oh   = F.one_hot(x.long(), num_classes=10).float()   # (B, 81, 10)
        x_grid = x_oh.view(B, 9, 9, 10).permute(0, 3, 1, 2)   # (B, 10, 9, 9)
        out    = self.cnn(x_grid)                               # (B, 9, 9, 9)
        return out.permute(0, 2, 3, 1).contiguous().view(B, 81, 9)

    @property
    def name(self): return 'CNN'

print('LegacyCNNSolver defined.')

## 2 — Load Test Data

In [ ]:
raw = np.load(DATA_PATH)
puzzles   = torch.tensor(raw['puzzles'].astype(np.int64),   dtype=torch.long)
solutions = torch.tensor(raw['solutions'].astype(np.int64), dtype=torch.long)

test_ds     = TensorDataset(puzzles, solutions)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f'Test puzzles : {len(puzzles):,}')
print(f'Puzzle shape : {puzzles.shape}  |  Solutions shape : {solutions.shape}')
print(f'Digit range  : {solutions.min().item()} – {solutions.max().item()}')

## 3 — Load All Models (with key-remap fixes)

In [ ]:
def _load_state(path):
    state = torch.load(path, map_location=DEVICE)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    return state

def _remap(state, old_prefix, new_prefix):
    """Rename keys: old_prefix.* → new_prefix.*"""
    return {
        (new_prefix + k[len(old_prefix):] if k.startswith(old_prefix) else k): v
        for k, v in state.items()
    }

MODEL_DEFS = [
    ('MLP',    MLPSolver()),
    ('CNN',    LegacyCNNSolver()),   # uses legacy arch matching saved weights
    ('RNN',    RNNSolver()),
    ('LSTM',   LSTMSolver()),
    ('GRU',    GRUSolver()),
    ('Hybrid', HybridSolver()),
]

models = {}
for name, model in MODEL_DEFS:
    state = _load_state(WEIGHTS_DIR / f'{name.lower()}.pt')

    # Fix key mismatches caused by architecture refactors
    if name in ('RNN', 'LSTM', 'GRU'):
        state = _remap(state, 'emb.', 'embedding.')
    elif name == 'Hybrid':
        state = _remap(state, 'cnn.', 'spatial_cnn.')

    model.load_state_dict(state, strict=True)
    model.to(DEVICE).eval()
    models[name] = model
    print(f'  {name:8} OK')

print('\nAll models loaded.')

## 4 — Evaluation Function

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    """
    Returns:
      all_preds   : (N*81,) int array — predicted digit 1-9
      all_labels  : (N*81,) int array — true digit 1-9
      puzzle_ok   : (N,)    bool array — True if entire puzzle correct
    """
    preds_list, labels_list, puzzle_ok = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)                  # (B, 81, 9)
        pred   = logits.argmax(dim=-1) + 1  # class 0..8 → digit 1..9
        preds_list.append(pred.cpu().numpy())
        labels_list.append(yb.cpu().numpy())
        puzzle_ok.append((pred == yb).all(dim=-1).cpu().numpy())

    return (
        np.concatenate(preds_list).reshape(-1),
        np.concatenate(labels_list).reshape(-1),
        np.concatenate(puzzle_ok),
    )

## 5 — Run Evaluation on All Models

In [ ]:
results = {}
digits  = list(range(1, 10))

for name, model in models.items():
    preds, labels, puzzle_ok = evaluate(model, test_loader)

    cell_acc    = (preds == labels).mean()
    puzzle_acc  = puzzle_ok.mean()
    f1_macro    = f1_score(labels, preds, average='macro',    labels=digits)
    f1_weighted = f1_score(labels, preds, average='weighted', labels=digits)
    f1_per_dig  = f1_score(labels, preds, average=None,       labels=digits)

    results[name] = dict(
        cell_acc=cell_acc, puzzle_acc=puzzle_acc,
        f1_macro=f1_macro, f1_weighted=f1_weighted,
        f1_per_dig=f1_per_dig, preds=preds, labels=labels,
    )

    print(f"{name:8} | cell={cell_acc:.4f} | puzzle={puzzle_acc:.4f} "
          f"| F1_macro={f1_macro:.4f} | F1_weighted={f1_weighted:.4f}")

## 6 — Summary Table

In [ ]:
rows = [
    {'Model': name,
     'Cell Acc':    f"{r['cell_acc']:.4f}",
     'Puzzle Acc':  f"{r['puzzle_acc']:.4f}",
     'F1 Macro':    f"{r['f1_macro']:.4f}",
     'F1 Weighted': f"{r['f1_weighted']:.4f}"}
    for name, r in results.items()
]
df = pd.DataFrame(rows).set_index('Model')
display(df.style
    .highlight_max(axis=0, color='#d4edda')
    .highlight_min(axis=0, color='#f8d7da'))

## 7 — F1 Score Per Digit (1–9)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for ax, (name, r) in zip(axes, results.items()):
    bars = ax.bar(digits, r['f1_per_dig'], color='steelblue', edgecolor='white')
    ax.set_title(f"{name}  (macro F1={r['f1_macro']:.3f})", fontsize=12, fontweight='bold')
    ax.set_xlabel('Digit')
    ax.set_ylabel('F1 Score')
    ax.set_xticks(digits)
    ax.set_ylim(0, 1)
    ax.axhline(r['f1_macro'], color='tomato', linestyle='--', linewidth=1.2, label='macro avg')
    ax.legend(fontsize=8)
    for bar, val in zip(bars, r['f1_per_dig']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=7)

plt.suptitle('F1 Score per Digit — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('weights/f1_per_digit.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → weights/f1_per_digit.png')

## 8 — Confusion Matrix (Best Model by F1)

In [ ]:
best_name = max(results, key=lambda k: results[k]['f1_macro'])
r  = results[best_name]
cm = confusion_matrix(r['labels'], r['preds'], labels=digits)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=digits, yticklabels=digits, ax=axes[0])
axes[0].set_title(f'{best_name} — Confusion Matrix (counts)', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=digits, yticklabels=digits, ax=axes[1])
axes[1].set_title(f'{best_name} — Confusion Matrix (normalised)', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig('weights/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best model: {best_name}  |  Saved → weights/confusion_matrix.png')

## 9 — Full Classification Report (Best Model)

In [ ]:
r = results[best_name]
print(f'=== {best_name} — Full Classification Report ===')
print(classification_report(r['labels'], r['preds'],
                            labels=digits,
                            target_names=[f'digit {d}' for d in digits]))

## 10 — Training History: Loss Curves

In [ ]:
model_tags   = ['mlp', 'cnn', 'rnn', 'lstm', 'gru', 'hybrid']
histories    = {m.upper(): json.load(open(WEIGHTS_DIR / f'{m}_history.json')) for m in model_tags}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (name, h) in zip(axes.flatten(), histories.items()):
    ep = range(1, len(h['train_loss']) + 1)
    ax.plot(ep, h['train_loss'], label='Train', color='steelblue')
    ax.plot(ep, h['val_loss'],   label='Val',   color='tomato')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(fontsize=8)
    ax.set_xlim(1, len(h['train_loss']))

plt.suptitle('Training vs Validation Loss', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('weights/loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 11 — Training History: Cell Accuracy Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (name, h) in zip(axes.flatten(), histories.items()):
    ep = range(1, len(h['train_cell_acc']) + 1)
    ax.plot(ep, h['train_cell_acc'], label='Train', color='steelblue')
    ax.plot(ep, h['val_cell_acc'],   label='Val',   color='tomato')
    # Draw test accuracy as a horizontal line
    matches = [k for k in results if k.upper() == name]
    if matches:
        tv = results[matches[0]]['cell_acc']
        ax.axhline(tv, color='green', linestyle=':', linewidth=1.5, label=f'Test={tv:.3f}')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Cell Accuracy')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
    ax.set_xlim(1, len(h['train_cell_acc']))

plt.suptitle('Cell Accuracy — Train / Val / Test', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('weights/accuracy_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 12 — Bar Chart: All Metrics Side-by-Side

In [ ]:
names_plot  = list(results.keys())
cell_accs   = [results[m]['cell_acc']    for m in names_plot]
puzzle_accs = [results[m]['puzzle_acc']  for m in names_plot]
f1_macros   = [results[m]['f1_macro']    for m in names_plot]
f1_weighted = [results[m]['f1_weighted'] for m in names_plot]

x, w = np.arange(len(names_plot)), 0.2
fig, ax = plt.subplots(figsize=(13, 6))
b1 = ax.bar(x - 1.5*w, cell_accs,   w, label='Cell Accuracy',  color='#4C72B0')
b2 = ax.bar(x - 0.5*w, puzzle_accs, w, label='Puzzle Accuracy', color='#DD8452')
b3 = ax.bar(x + 0.5*w, f1_macros,   w, label='F1 Macro',        color='#55A868')
b4 = ax.bar(x + 1.5*w, f1_weighted, w, label='F1 Weighted',     color='#C44E52')

ax.set_xticks(x); ax.set_xticklabels(names_plot, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — All Metrics', fontsize=13, fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
for bars in [b1, b2, b3, b4]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f'{h:.2f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('weights/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → weights/model_comparison.png')

## 13 — Final Interpretation

In [ ]:
print('=' * 62)
print('  EVALUATION SUMMARY')
print('=' * 62)
print(f"  {'Model':8}  {'Cell Acc':>9}  {'Puzzle Acc':>10}  {'F1 Macro':>8}  Grade")
print('-' * 62)
for name, r in results.items():
    ca = r['cell_acc']
    grade = ('EXCELLENT' if ca >= 0.90 else
             'GOOD'      if ca >= 0.75 else
             'MODERATE'  if ca >= 0.50 else 'POOR')
    print(f"  {name:8}  {ca:>9.4f}  {r['puzzle_acc']:>10.4f}  "
          f"{r['f1_macro']:>8.4f}  {grade}")
print('=' * 62)
best_f1   = max(results, key=lambda k: results[k]['f1_macro'])
best_cell = max(results, key=lambda k: results[k]['cell_acc'])
best_puz  = max(results, key=lambda k: results[k]['puzzle_acc'])
print(f'  Best F1 Macro   : {best_f1}   ({results[best_f1]["f1_macro"]:.4f})')
print(f'  Best Cell Acc   : {best_cell}  ({results[best_cell]["cell_acc"]:.4f})')
print(f'  Best Puzzle Acc : {best_puz}  ({results[best_puz]["puzzle_acc"]:.4f})')

---
## 14 — Solve a Real Puzzle with All 6 Models

In [ ]:
# ── Run every model and display results ──────────────────────────────────────
n_models = len(models)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

# First subplot = input puzzle
draw_sudoku(axes[0], puzzle_np, clue_mask, title='INPUT PUZZLE')

solve_summary = []

for idx, (name, model) in enumerate(models.items(), start=1):
    with torch.no_grad():
        logits = model(puzzle_flat)                    # (1, 81, 9)
        probs  = torch.softmax(logits, dim=-1)
        pred   = logits.argmax(dim=-1).squeeze() + 1  # (81,) digits 1-9

    pred_grid = pred.cpu().numpy().reshape(9, 9)

    # Confidence = mean max-prob across empty cells only
    max_p = probs.squeeze().max(dim=-1).values.cpu().numpy().reshape(9, 9)
    conf  = max_p[~clue_mask].mean()

    # Fill clues back (model predicts all 81 cells; trust given clues)
    display_grid = pred_grid.copy()
    display_grid[clue_mask] = puzzle_np[clue_mask]

    # Check validity
    valid = is_valid_sudoku(display_grid)

    # Highlight cells where model disagrees with a given clue
    wrong_clue = clue_mask & (pred_grid != puzzle_np)

    title = f'{name}\nconf={conf:.2f}  valid={"✓" if valid else "✗"}'
    draw_sudoku(axes[idx], display_grid, clue_mask,
                title=title, highlight_wrong=wrong_clue)

    solve_summary.append({
        'Model': name,
        'Confidence': f'{conf:.3f}',
        'Valid': '✓' if valid else '✗',
        'Clue Errors': int(wrong_clue.sum()),
    })

# Hide unused subplot
for ax in axes[n_models+1:]:
    ax.axis('off')

plt.suptitle('All 6 Models — Solving the Puzzle\n'
             '(blue=given clue, green=predicted, red=model disagreed with clue)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('weights/puzzle_solve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → weights/puzzle_solve.png')

# Summary table
print()
df_solve = pd.DataFrame(solve_summary).set_index('Model')
display(df_solve)

In [ ]:
# ── Puzzle read from the image (0 = empty cell) ──────────────────────────────
MY_PUZZLE = [
    [1, 0, 0, 0, 3, 4, 0, 0, 8],
    [0, 7, 0, 6, 8, 0, 0, 3, 0],
    [0, 0, 8, 2, 1, 0, 7, 0, 4],
    [0, 5, 4, 0, 9, 0, 6, 8, 0],
    [9, 1, 0, 5, 0, 8, 0, 2, 0],
    [0, 8, 0, 3, 0, 0, 0, 0, 5],
    [3, 0, 5, 9, 0, 6, 8, 7, 1],
    [0, 0, 6, 0, 0, 0, 0, 4, 0],
    [0, 0, 1, 0, 7, 0, 2, 0, 0],
]

# ── Helper: check Sudoku validity ────────────────────────────────────────────
def is_valid_sudoku(grid):
    """Returns True if 9×9 grid satisfies all row/col/box constraints."""
    g = np.array(grid)
    for i in range(9):
        for vals in [g[i, :], g[:, i],
                     g[3*(i//3):3*(i//3)+3, 3*(i%3):3*(i%3)+3].flatten()]:
            s = sorted(vals)
            if s != list(range(1, 10)):
                return False
    return True

# ── Helper: draw a sudoku grid ───────────────────────────────────────────────
def draw_sudoku(ax, grid, clue_mask, title='', highlight_wrong=None):
    """
    grid       : 9×9 int array (1-9)
    clue_mask  : 9×9 bool — True where cell was a given clue
    highlight_wrong: 9×9 bool or None — True where prediction disagrees
    """
    ax.set_xlim(0, 9); ax.set_ylim(0, 9)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=10, fontweight='bold', pad=4)

    # Background colours
    for r in range(9):
        for c in range(9):
            if highlight_wrong is not None and highlight_wrong[r, c]:
                color = '#ffcccc'   # red  — wrong cell
            elif clue_mask[r, c]:
                color = '#dce8f5'   # blue — given clue
            else:
                color = 'white'
            ax.add_patch(plt.Rectangle((c, 8-r), 1, 1,
                                       facecolor=color, edgecolor='#aaaaaa', linewidth=0.5))

    # Bold box borders
    for i in range(10):
        lw = 2.5 if i % 3 == 0 else 0.5
        ax.plot([i, i], [0, 9], 'k-', linewidth=lw)
        ax.plot([0, 9], [i, i], 'k-', linewidth=lw)

    # Digits
    for r in range(9):
        for c in range(9):
            v = grid[r, c]
            if v != 0:
                color = '#1a1a2e' if clue_mask[r, c] else '#2d6a4f'
                ax.text(c + 0.5, 8 - r + 0.5, str(v),
                        ha='center', va='center', fontsize=11,
                        fontweight='bold' if clue_mask[r, c] else 'normal',
                        color=color)

# ── Prepare input tensor ─────────────────────────────────────────────────────
puzzle_np   = np.array(MY_PUZZLE, dtype=np.int64)         # (9,9)
clue_mask   = puzzle_np != 0                               # True = given
puzzle_flat = torch.tensor(puzzle_np.reshape(1, 81), dtype=torch.long).to(DEVICE)

clues_count = clue_mask.sum()
print(f'Puzzle loaded — {clues_count} clues given, {81 - clues_count} cells to predict')
print()
print('Input puzzle:')
for row in MY_PUZZLE:
    print(' '.join(str(x) if x else '.' for x in row))